# Expiry Day Volatility Analysis

This notebook analyzes whether NIFTY option expiry days experience significantly different volatility compared to normal trading days.

We will calculate four volatility proxies for each day:
1. **Intraday Range**: `(High - Low) / Open`
2. **Absolute Daily Return**: `|Close - Previous Close| / Previous Close`
3. **Parkinson Variance**: `(1 / (4 * ln(2))) * [ln(High / Low)]^2`
4. **Garman-Klass Variance**: `0.5 * [ln(High / Low)]^2 - (2*ln(2) - 1) * [ln(Close / Open)]^2`

*Note: For Parkinson and Garman-Klass, we work in variance space to avoid Jensen's inequality (averaging standard deviations directly is biased).*


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# 1. Load Data
df = pd.read_csv('../data/processed/data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# We only need the OHLC data for the NIFTY index itself to measure index volatility
df = df[df['Symbol'] == 'NIFTY'].copy()

# Sort by date for previous close calculations
df = df.sort_values('Date').reset_index(drop=True)

# Load Expiry Dates
expiries_df = pd.read_csv('../data/processed/nifty_actual_expiries.csv')
expiries_df['ExpiryDate'] = pd.to_datetime(expiries_df['ExpiryDate'])

# Create a set of expiry dates for fast lookup
expiry_dates = set(expiries_df['ExpiryDate'])

# Flag Expiry Days
df['is_expiry'] = df['Date'].isin(expiry_dates)

print(f"Total trading days: {len(df)}")
print(f"Normal days: {(~df['is_expiry']).sum()}")
print(f"Expiry days: {df['is_expiry'].sum()}")


In [ ]:
# 2. Calculate Volatility Metrics

# Previous Close
df['prev_close'] = df['close'].shift(1)
df = df.dropna(subset=['prev_close']).copy()

# Metric 1: Intraday Range
df['intraday_range'] = (df['high'] - df['low']) / df['open']

# Metric 2: Absolute Daily Return
df['abs_daily_return'] = np.abs(df['close'] - df['prev_close']) / df['prev_close']

# Log ratios
ln_hl = np.log(df['high'] / df['low'])
ln_co = np.log(df['close'] / df['open'])

# Metric 3: Parkinson Variance (Daily)
df['parkinson_var'] = (1 / (4 * np.log(2))) * (ln_hl ** 2)

# Metric 4: Garman-Klass Variance (Daily)
df['garman_klass_var'] = 0.5 * (ln_hl ** 2) - (2 * np.log(2) - 1) * (ln_co ** 2)

# Convert variance to annualized Volatility (assuming 252 trading days)
# We calculate this primarily for interpretable plotting
df['parkinson_vol_annualized'] = np.sqrt(df['parkinson_var'] * 252)
df['garman_klass_vol_annualized'] = np.sqrt(df['garman_klass_var'] * 252)

df.head()


In [ ]:
# 3. Exploratory Data Analysis (EDA)

metrics_to_plot = {
    'intraday_range': 'Intraday Range ((High-Low)/Open)',
    'abs_daily_return': 'Absolute Daily Return (|Close-PrevClose| / PrevClose)',
    'parkinson_var': 'Parkinson Variance (Daily)',
    'garman_klass_var': 'Garman-Klass Variance (Daily)'
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, (metric, title) in enumerate(metrics_to_plot.items()):
    sns.boxplot(x='is_expiry', y=metric, data=df, ax=axes[i], showfliers=False)
    axes[i].set_title(f'Comparison of {title}')
    axes[i].set_xticklabels(['Normal Day', 'Expiry Day'])
    axes[i].set_xlabel('')
    axes[i].set_ylabel(metric)

plt.tight_layout()
plt.show()


In [ ]:
# 4. Statistical Testing

# We will use the Mann-Whitney U test (non-parametric) as volatility distributions are highly right-skewed.
# Null Hypothesis: Distribution of volatility on Normal days = Distribution of volatility on Expiry days.

normal_days = df[~df['is_expiry']]
expiry_days = df[df['is_expiry']]

results = []

for metric, title in metrics_to_plot.items():
    # Mean calculations
    mean_normal = normal_days[metric].mean()
    mean_expiry = expiry_days[metric].mean()
    
    # Mann-Whitney U Test
    stat, p_val = stats.mannwhitneyu(normal_days[metric], expiry_days[metric], alternative='two-sided')
    
    # T-test (for reference, though U-test is more robust here)
    t_stat, t_pval = stats.ttest_ind(normal_days[metric], expiry_days[metric], equal_var=False)
    
    results.append({
        'Metric': title,
        'Mean (Normal)': mean_normal,
        'Mean (Expiry)': mean_expiry,
        'Ratio (Expiry/Normal)': mean_expiry / mean_normal if mean_normal != 0 else np.nan,
        'Mann-Whitney p-value': p_val,
        'T-test p-value': t_pval
    })

results_df = pd.DataFrame(results)
display(results_df)


### Conclusion

*   Look at the `Ratio (Expiry/Normal)` column. A value > 1 indicates average volatility was higher on expiry days.
*   Look at the `Mann-Whitney p-value`. If it is less than 0.05, the difference in volatility between the two groups is statistically significant.
*   By testing variance directly (for Parkinson and Garman-Klass), we ensure mathematical soundness before any potential conversion back to annualized standard deviations.
